# Pyspark vs SQL
regla mental que debes repetir en clase

-SQL = declarativo

-Pyspark = transformaciones encadenadas sobre dataframes

In [0]:
%sql
SELECT * FROM samples.bakehouse.sales_customers
LIMIT 5

In [0]:
df = spark.read.table("samples.bakehouse.sales_customers")

df_selected = df.select("*")##.limit(10)

In [0]:
df_selected.display()

##DISTINCT

In [0]:
%sql

SELECT DISTINCT country
FROM samples.bakehouse.sales_customers

In [0]:
df_distinct = df_selected.select("country").distinct()
display(df_distinct)

## WHERE / FILTER

In [0]:
%sql
SELECT *
FROM samples.bakehouse.sales_customers
WHERE 1=1
AND country = "Japan"
AND gender = "female"
LIMIT 5
;

In [0]:
from pyspark.sql.functions import col
df_filtered = df_selected.filter(
    (col("country")== "USA")&
    (col("gender")== "female")
)
display(df_filtered)

##ORDER BY + LIMIT

In [0]:
%sql
SELECT *
FROM samples.bakehouse.sales_customers
ORDER BY first_name ASC
LIMIT 15;

In [0]:
df_sorted = df.orderBy(col("first_name").desc())
display(df_sorted.limit(15))

## CREATE COLUMN

In [0]:
%sql 
SELECT "Hola" AS saludo

In [0]:
%sql
SELECT 
  concat(first_name, " ", last_name) AS full_name,
  concat_ws(', ' , city, state, country, continent) AS address,
  35 AS age,
  "random comment" AS comment,
  true AS is_true
FROM samples.bakehouse.sales_customers

LIMIT 10

In [0]:
from pyspark.sql.functions import col, concat, lit

df_plus_columns= df.select(
    concat(col("first_name"), lit(" "), col("last_name")).alias("full_name"),
    lit(True).alias("is_True"),
    lit("random_comment").alias("comment")
).limit(10)

display(df_plus_columns)

## GROUP BY + AGGREGATION

In [0]:
%sql
SELECT country, count(1) AS cantidad
FROM samples.bakehouse.sales_franchises
GROUP BY country
ORDER BY cantidad desc

In [0]:
from pyspark.sql.functions import count

df_grouped = (
    df
    .groupBy("country")
    .agg(count("*").alias("count"))
    .orderBy(col("count").desc())
)

display(df_grouped)

## JOIN (CLAVE PARA SILVER)

In [0]:
%sql
SELECT 
  concat(c.first_name, " ", c.last_name) AS full_name,
  c.country,
  c.gender,
  s.product,
  s.quantity,
  s.paymentMethod,
  s.dateTime
FROM samples.bakehouse.sales_transactions s
JOIN samples.bakehouse.sales_customers c
  ON s.customerID = c.customerID;

In [0]:
df_sales = spark.read.table("samples.bakehouse.sales_transactions")

df_joined =(df
          .join(df_sales, on="customerID", how="inner")
          .select(
              ("*")
          )
)

df_joined.display()

## CASTEAR

Castear en programación es el proceso de convertir una variable o dato de un tipo a otro (por ejemplo, de texto string a número int o entero a decimal float). Permite forzar la compatibilidad entre datos, ya sea de forma automática (implícita) o mediante código específico (explícita), siendo fundamental para la manipulación de datos.

In [0]:
%sql

SELECT 
  CAST(transactionID AS string) AS transactionID,
  CAST(customerID AS string) AS customerID,
  CAST(franchiseID AS string) AS franchiseID,
  CAST(dateTime AS string) AS dateTime,
  CAST(product AS string) AS product

FROM samples.bakehouse.sales_transactions;

In [0]:
df_casted= df_sales.select(
    col("transactionID").cast("bigint"),
    col("customerID").cast("string"),
    col("franchiseID").cast("string"),
    col("dateTime").cast("string"),
    col("product"),
    col("quantity").cast("bigint"),
    col("unitprice").cast("double"),

    (col("unitPrice")* col("quantity")).cast("bigint").alias("totalPrice"),
)

df_casted.display()

una forma de agregar o ajustar una columna sin "select" es con "withColumn", dado que con "select" toca escribir todas las columnas, de lo contrario te va a omitir la que no escribas. Sin embargo, se recomienda siempre hacer el select.

In [0]:
df_test = df_casted.withColumn("nuevaColumnita", lit("nuevaColumnita_literal!!!"))

df_test.display()

##DROP

In [0]:
df_new = df_test.drop("nuevaColumnita")
df_new.display()